# Lung Cancer T-cell scRNA-seq: Load, Extract, and Reprocess

Standalone pipeline (no external agent-framework dependency) that:

1. **Loads** an original single-cell h5ad dataset from one of three sources you choose in the Parameters cell:
   - a **CELLxGENE Census** `dataset_id` (queries and downloads only the matching cells, e.g. the LuCA lung cancer atlas)
   - a **remote URL** to an `.h5ad` file (e.g. a CELLxGENE Explorer "Download Dataset" link)
   - a **local file path** to an `.h5ad` file already on disk
2. **Extracts T cells** from that data (by `cell_type` obs column, if the loaded object contains more than just T cells)
3. **Reprocesses** from raw counts: QC filtering -> normalization -> HVG/PCA -> Harmony batch correction -> neighbor graph -> UMAP -> Leiden clustering
4. **Annotates** clusters into biological T-cell subsets (naive/Tcm, Treg, effector memory, terminal effector, exhausted, proliferating, etc.)
5. **Visualizes**: UMAP colored by subset, a marker-gene heatmap, and a marker-gene dot plot

Every function is defined directly in this notebook. It depends only on
`scanpy`, `harmonypy`, `cellxgene_census`, `pandas`, `numpy`, `scipy`,
`matplotlib`, and `requests` — no Biomni or other agent-framework import.


## 1. Imports

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1


## 2. Parameters — edit this cell for your run

Set `DATA_SOURCE` to one of `"census"`, `"url"`, or `"local"`, and fill in the
matching value below. Everything downstream reads from these variables only.

In [ ]:
# --- Choose ONE data source -------------------------------------------------
DATA_SOURCE = "census"   # "census" | "url" | "local"

# Used when DATA_SOURCE == "census": a CELLxGENE Census dataset_id (UUID).
# Example below is the LuCA lung cancer atlas (Salcher et al.).
CENSUS_DATASET_ID = "232f6a5a-a04c-4758-a6e8-88ab2e3a6e69"
CENSUS_VERSION = "stable"

# Used when DATA_SOURCE == "url": a direct HTTP(S) link to an .h5ad file
# (e.g. a CELLxGENE Explorer "Download Dataset" URL -- NOT a browse-page URL).
REMOTE_H5AD_URL = "https://datasets.cellxgene.cziscience.com/<uuid>.h5ad"

# Used when DATA_SOURCE == "local": a path to an .h5ad file already on disk.
LOCAL_H5AD_PATH = "luca_tcells_raw.h5ad"

# --- T-cell extraction filter (applied after loading, if the obs column exists) ---
CELL_TYPE_COLUMN = "cell_type"
T_CELL_TYPES = [
    "CD4-positive, alpha-beta T cell",
    "CD8-positive, alpha-beta T cell",
    "regulatory T cell",
    "T cell",
]
TISSUE_COLUMN = "tissue"
TISSUE_FILTER = None          # e.g. "lung"; None = no tissue filter
DISEASE_COLUMN = "disease"
BALANCE_BY = "disease"        # obs column to balance downsampling across, or None
MAX_CELLS_PER_GROUP = 3500    # cap per BALANCE_BY group; used for "census" source
MAX_CELLS_TOTAL = None        # uniform cap on total cells; used for "url"/"local" sources
RANDOM_SEED = 0

# --- QC / processing parameters --------------------------------------------
BATCH_KEY = "assay"                 # obs column for Harmony batch correction; None to skip
MIN_GENES = 200
MIN_CELLS = 3
MAX_TOTAL_COUNTS_QUANTILE = 0.995
DROP_ASSAY_VALUES = ["Smart-seq2"]  # non-UMI assays, not magnitude-comparable; [] to keep all
N_TOP_GENES = 2000
N_PCS = 50
N_NEIGHBORS = 15
LEIDEN_RESOLUTION = 1.0

# --- Output paths ------------------------------------------------------------
RAW_H5AD_OUT = "notebook_tcells_raw.h5ad"
PROCESSED_H5AD_OUT = "notebook_tcells_processed.h5ad"
ANNOTATED_H5AD_OUT = "notebook_tcells_annotated.h5ad"
UMAP_PNG_OUT = "notebook_umap_by_subset.png"
HEATMAP_PNG_OUT = "notebook_marker_heatmap.png"
DOTPLOT_PNG_OUT = "notebook_marker_dotplot.png"

# --- Marker gene panel for visualization -------------------------------------
MARKER_GENES = [
    "CCR7", "SELL", "TCF7", "IL7R",                 # naive / Tcm
    "FOXP3", "CTLA4", "IL2RA",                      # regulatory T
    "GZMK", "GZMA",                                 # effector memory
    "GZMH", "GZMB", "GNLY", "NKG7", "PRF1",         # cytotoxic / terminal effector
    "CXCL13", "HAVCR2", "PDCD1", "LAG3", "TIGIT",   # exhaustion
    "MKI67",                                        # proliferation
    "KLRB1", "KLRC1",                               # innate-like
]

print(f"DATA_SOURCE = {DATA_SOURCE!r}")


## 3. Load the original data

Exactly one of the three branches below runs, based on `DATA_SOURCE`.

In [ ]:
if DATA_SOURCE == "census":
    import cellxgene_census

    census = cellxgene_census.open_soma(census_version=CENSUS_VERSION)
    obs_df = cellxgene_census.get_obs(
        census, "homo_sapiens",
        value_filter=f"dataset_id == '{CENSUS_DATASET_ID}'",
        column_names=["soma_joinid", CELL_TYPE_COLUMN, DISEASE_COLUMN, TISSUE_COLUMN, "assay", "sex"],
    )
    print(f"Dataset {CENSUS_DATASET_ID}: {len(obs_df)} total cells available.")

elif DATA_SOURCE == "url":
    import requests

    raw_path = LOCAL_H5AD_PATH if 'LOCAL_H5AD_PATH' in dir() else "downloaded.h5ad.tmp"
    raw_path = "downloaded_original.h5ad"
    with requests.get(REMOTE_H5AD_URL, stream=True, timeout=1800) as r:
        r.raise_for_status()
        content_type = r.headers.get("content-type", "")
        if "html" in content_type.lower():
            raise ValueError(
                f"URL {REMOTE_H5AD_URL!r} returned content-type {content_type!r} -- looks like an "
                "HTML landing page, not a direct h5ad download link."
            )
        with open(raw_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    print(f"Downloaded {os.path.getsize(raw_path)/1e6:.1f} MB from {REMOTE_H5AD_URL} to {raw_path}")

elif DATA_SOURCE == "local":
    if not os.path.exists(LOCAL_H5AD_PATH):
        raise FileNotFoundError(f"LOCAL_H5AD_PATH not found: {LOCAL_H5AD_PATH}")
    print(f"Using local file: {LOCAL_H5AD_PATH} ({os.path.getsize(LOCAL_H5AD_PATH)/1e6:.1f} MB)")

else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE!r}")


## 4. Extract T cells and build the raw AnnData

For the `"census"` source, cells are balance-downsampled across `BALANCE_BY`
(e.g. per-disease-group) at fetch time, so no single group dominates
downstream clustering. For `"url"`/`"local"` sources the whole file is
loaded first, then filtered/optionally downsampled uniformly.

In [ ]:
if DATA_SOURCE == "census":
    mask = obs_df[CELL_TYPE_COLUMN].isin(T_CELL_TYPES)
    if TISSUE_FILTER is not None:
        mask &= obs_df[TISSUE_COLUMN].astype(str) == TISSUE_FILTER
    sub = obs_df[mask].copy()
    n_before = len(sub)

    np.random.seed(RANDOM_SEED)
    if BALANCE_BY is not None:
        parts = []
        for _, g in sub.groupby(sub[BALANCE_BY].astype(str)):
            n = min(MAX_CELLS_PER_GROUP, len(g))
            parts.append(g.sample(n=n, random_state=RANDOM_SEED))
        sel = pd.concat(parts)
    else:
        n = min(MAX_CELLS_PER_GROUP, len(sub)) if MAX_CELLS_PER_GROUP else len(sub)
        sel = sub.sample(n=n, random_state=RANDOM_SEED)

    adata = cellxgene_census.get_anndata(
        census, organism="homo_sapiens",
        obs_coords=sel.soma_joinid.tolist(),
        column_names={"obs": ["soma_joinid", CELL_TYPE_COLUMN, DISEASE_COLUMN, TISSUE_COLUMN,
                                "assay", "sex", "dataset_id", "self_reported_ethnicity", "donor_id"]},
    )
    print(f"Matched {n_before} T cells; balance-downsampled to {adata.n_obs} cells x {adata.n_vars} genes.")

else:
    src_path = raw_path if DATA_SOURCE == "url" else LOCAL_H5AD_PATH
    adata = sc.read_h5ad(src_path)
    n_loaded = adata.n_obs
    print(f"Loaded {n_loaded} cells x {adata.n_vars} genes from {src_path}.")

    if CELL_TYPE_COLUMN in adata.obs.columns:
        adata = adata[adata.obs[CELL_TYPE_COLUMN].astype(str).isin(T_CELL_TYPES)].copy()
        print(f"Extracted {adata.n_obs} T cells by '{CELL_TYPE_COLUMN}' (from {n_loaded}).")
    else:
        print(f"WARNING: '{CELL_TYPE_COLUMN}' not in obs columns -- assuming file already contains only T cells.")

    if TISSUE_FILTER is not None and TISSUE_COLUMN in adata.obs.columns:
        adata = adata[adata.obs[TISSUE_COLUMN].astype(str) == TISSUE_FILTER].copy()
        print(f"After tissue filter '{TISSUE_FILTER}': {adata.n_obs} cells.")

    if MAX_CELLS_TOTAL is not None and adata.n_obs > MAX_CELLS_TOTAL:
        rng = np.random.default_rng(RANDOM_SEED)
        idx = np.sort(rng.choice(adata.n_obs, size=MAX_CELLS_TOTAL, replace=False))
        adata = adata[idx].copy()
        print(f"Downsampled to {adata.n_obs} cells (MAX_CELLS_TOTAL={MAX_CELLS_TOTAL}).")

    if DATA_SOURCE == "url" and os.path.exists(raw_path):
        os.remove(raw_path)  # drop the full downloaded file once the T-cell subset is extracted

adata.write_h5ad(RAW_H5AD_OUT)
print(f"\nWrote raw T-cell AnnData to {RAW_H5AD_OUT}: {adata.n_obs} cells x {adata.n_vars} genes.")
adata


## 5. QC, normalization, batch correction, clustering, UMAP

Filters cells/genes, drops top total-count outliers and (optionally)
non-UMI assays, log-normalizes, selects highly variable genes, runs PCA,
corrects for batch effects with Harmony (if `BATCH_KEY` is set), builds a
neighbor graph on the batch-corrected space, and computes UMAP + Leiden
clusters.

In [ ]:
adata.layers["counts"] = adata.X.copy()

# Use gene symbols as var_names if a feature_name column is present (Census convention)
if "feature_name" in adata.var.columns:
    adata.var_names = adata.var["feature_name"].astype(str)
    adata.var_names_make_unique()
    adata.var.index.name = None  # avoid feature_name being both index and column at write time

adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None)

n0 = adata.n_obs
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)

upper = adata.obs.total_counts.quantile(MAX_TOTAL_COUNTS_QUANTILE)
adata = adata[adata.obs.total_counts <= upper].copy()

if DROP_ASSAY_VALUES and "assay" in adata.obs.columns:
    adata = adata[~adata.obs.assay.astype(str).isin(DROP_ASSAY_VALUES)].copy()

print(f"QC: {n0} -> {adata.n_obs} cells retained "
      f"(min_genes={MIN_GENES}, total_counts<={MAX_TOTAL_COUNTS_QUANTILE:.3f} quantile"
      f"{', assay filter applied' if DROP_ASSAY_VALUES else ''}).")


In [ ]:
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

hvg_kwargs = {"n_top_genes": N_TOP_GENES}
if BATCH_KEY is not None and BATCH_KEY in adata.obs.columns:
    hvg_kwargs["batch_key"] = BATCH_KEY
sc.pp.highly_variable_genes(adata, **hvg_kwargs)

adata.raw = adata  # keep full log-normalized matrix for marker plotting later

adata_hvg = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=N_PCS, svd_solver="arpack")
adata.obsm["X_pca"] = adata_hvg.obsm["X_pca"]

print(f"Selected {adata.var.highly_variable.sum()} HVGs; computed {N_PCS}-component PCA.")


In [ ]:
rep = "X_pca"
if BATCH_KEY is not None and BATCH_KEY in adata.obs.columns:
    import harmonypy

    ho = harmonypy.run_harmony(adata.obsm["X_pca"], adata.obs, [BATCH_KEY], max_iter_harmony=20)
    Zc = np.asarray(ho.Z_corr)
    adata.obsm["X_pca_harmony"] = Zc.T if Zc.shape[0] != adata.n_obs else Zc
    rep = "X_pca_harmony"
    print(f"Harmony batch correction applied on '{BATCH_KEY}'.")
else:
    print("Skipping Harmony batch correction (BATCH_KEY not set or not in obs).")

sc.pp.neighbors(adata, use_rep=rep, n_neighbors=N_NEIGHBORS)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, key_added="leiden", flavor="igraph", n_iterations=2)

cluster_counts = adata.obs.leiden.value_counts().sort_index()
print(f"Leiden (resolution={LEIDEN_RESOLUTION}) found {cluster_counts.shape[0]} clusters:")
print(cluster_counts)

adata.write_h5ad(PROCESSED_H5AD_OUT)
print(f"\nWrote processed AnnData (with UMAP + Leiden) to {PROCESSED_H5AD_OUT}.")


## 6. Inspect clusters, then annotate biological subsets

Run this cell to see each cluster's size and mean expression of a few
canonical markers, to help you fill in `LABEL_MAP` below (or adjust the
default mapping if your cluster numbering differs from the reference run).

In [ ]:
inspect_genes = [g for g in MARKER_GENES if g in adata.raw.var_names]
X_inspect = adata.raw[:, inspect_genes].X
X_inspect = X_inspect.toarray() if hasattr(X_inspect, "toarray") else X_inspect
inspect_df = pd.DataFrame(X_inspect, columns=inspect_genes, index=adata.obs_names)
inspect_df["leiden"] = adata.obs["leiden"].values
cluster_profile = inspect_df.groupby("leiden", observed=True).mean()
cluster_profile["n_cells"] = adata.obs["leiden"].value_counts()
cluster_profile


Edit `LABEL_MAP` below to match what you see above. Any cluster ID present
in the data but missing from this dict is auto-labeled
`"Cluster <id> (unannotated)"` rather than dropped, so re-running at a
different resolution never silently loses cells. List clusters you judge to
be low-quality (ambient RNA / doublets / stress response / high-mito) in
`LOW_QUALITY_CLUSTERS` to flag them without deleting them.

In [ ]:
LABEL_MAP = {
    "0": "CD4 T resting",
    "1": "CD4 Naive/Tcm",
    "3": "Regulatory T (Treg)",
    "4": "CD8 Effector Memory (GZMK+)",
    "6": "CD8 Terminal Effector (GZMH+GNLY+)",
    "7": "CD8 Memory (NK-like, KLRC1+)",
    "8": "CD8 Exhausted/Tumor-reactive (CXCL13+GZMB+)",
    "9": "Proliferating T (MKI67+)",
    "10": "CD4/Treg CXCL13+ (dysfunctional, tumor-assoc.)",
    "11": "Innate-like T (MAIT/NKT-like, KLRB1+)",
    # ... add/adjust entries based on the cluster_profile table above
}
LOW_QUALITY_CLUSTERS = ["2", "5", "13", "14", "16"]  # ambient RNA / doublets / stress / high-mito

present_clusters = adata.obs["leiden"].astype(str).unique()
full_map = dict(LABEL_MAP)
for c in present_clusters:
    full_map.setdefault(c, f"Cluster {c} (unannotated)")

adata.obs["subset"] = adata.obs["leiden"].astype(str).map(full_map).astype("category")
adata.obs["is_low_quality"] = adata.obs["leiden"].astype(str).isin(LOW_QUALITY_CLUSTERS)

subset_counts = adata.obs["subset"].value_counts()
print(f"Annotated {adata.n_obs} cells into {subset_counts.shape[0]} subsets:")
print(subset_counts)
print(f"\nFlagged {int(adata.obs['is_low_quality'].sum())} cells as low quality.")

adata.write_h5ad(ANNOTATED_H5AD_OUT)
print(f"\nWrote annotated AnnData to {ANNOTATED_H5AD_OUT}.")


## 7. Visualize: UMAP colored by subset (excluding low-quality clusters)

In [ ]:
adata_clean = adata[~adata.obs["is_low_quality"]].copy()
print(f"Plotting {adata_clean.n_obs} clean cells (excluded {int(adata.obs['is_low_quality'].sum())} low-quality).")

groups = adata_clean.obs["subset"].astype(str).value_counts().index.tolist()
palette = plt.get_cmap("tab20")(np.linspace(0, 1, len(groups)))
color_map = dict(zip(groups, palette))
coords = adata_clean.obsm["X_umap"]

fig, ax = plt.subplots(figsize=(8, 6.5))
for g in groups:
    m = adata_clean.obs["subset"].astype(str) == g
    ax.scatter(coords[m, 0], coords[m, 1], s=3, alpha=0.7, color=color_map[g], label=g, linewidths=0)
ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
ax.set_title("T-cell subsets", loc="left")
ax.legend(fontsize=6, markerscale=3, loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.savefig(UMAP_PNG_OUT, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {UMAP_PNG_OUT}")


## 8. Visualize: marker gene heatmap (z-scored mean expression per subset)

In [ ]:
genes_used = [g for g in MARKER_GENES if g in adata_clean.raw.var_names]
skipped = set(MARKER_GENES) - set(genes_used)
if skipped:
    print(f"Skipped (not found in data): {sorted(skipped)}")

X = adata_clean.raw[:, genes_used].X
expr = pd.DataFrame(X.toarray() if hasattr(X, "toarray") else X, columns=genes_used, index=adata_clean.obs_names)
expr["subset"] = adata_clean.obs["subset"].values
mean_expr = expr.groupby("subset", observed=True).mean()

order = adata_clean.obs["subset"].value_counts().index
mean_expr = mean_expr.loc[[g for g in order if g in mean_expr.index]]
zscored = (mean_expr - mean_expr.mean(axis=0)) / (mean_expr.std(axis=0) + 1e-9)

fig, ax = plt.subplots(figsize=(max(6, 0.4 * len(genes_used)), max(4, 0.3 * len(zscored))))
im = ax.imshow(zscored.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(genes_used))); ax.set_xticklabels(genes_used, rotation=90, fontsize=7)
ax.set_yticks(range(len(zscored))); ax.set_yticklabels(zscored.index, fontsize=7)
ax.set_title("Marker expression by subset (z-score)", loc="left")
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("z-score", fontsize=7)
fig.tight_layout()
fig.savefig(HEATMAP_PNG_OUT, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {HEATMAP_PNG_OUT}")


## 9. Visualize: marker gene dot plot (size = % expressing, color = mean expression)

In [ ]:
pct_expr = (expr.drop(columns="subset") > 0).groupby(expr["subset"], observed=True).mean() * 100
pct_expr = pct_expr.loc[mean_expr.index]

fig, ax = plt.subplots(figsize=(max(6, 0.4 * len(genes_used)), max(4, 0.3 * len(mean_expr))))
xs, ys, sizes, colors = [], [], [], []
for i, s in enumerate(mean_expr.index):
    for j, g in enumerate(genes_used):
        xs.append(j); ys.append(i)
        sizes.append(pct_expr.loc[s, g] * 6)
        colors.append(mean_expr.loc[s, g])
sc_plot = ax.scatter(xs, ys, s=sizes, c=colors, cmap="viridis", edgecolors="black", linewidths=0.3)
ax.set_xticks(range(len(genes_used))); ax.set_xticklabels(genes_used, rotation=90, fontsize=7)
ax.set_yticks(range(len(mean_expr))); ax.set_yticklabels(mean_expr.index, fontsize=7)
ax.invert_yaxis()
ax.set_title("Marker expression dot plot by subset", loc="left")
for pct_val in [10, 30, 60]:
    ax.scatter([], [], s=pct_val * 6, c="grey", edgecolors="black", linewidths=0.3, label=f"{pct_val}%")
ax.legend(title="% expressing", loc="upper left", bbox_to_anchor=(1.12, 1.0), fontsize=7, title_fontsize=7.5, frameon=False)
cbar = fig.colorbar(sc_plot, ax=ax, fraction=0.03, pad=0.09)
cbar.set_label("mean expr (all cells)", fontsize=7)
fig.tight_layout()
fig.savefig(DOTPLOT_PNG_OUT, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {DOTPLOT_PNG_OUT}")


## 10. Summary

Outputs written by this notebook:
- `notebook_tcells_raw.h5ad` — extracted raw T cells
- `notebook_tcells_processed.h5ad` — QC'd, clustered, UMAP-embedded
- `notebook_tcells_annotated.h5ad` — with `subset` and `is_low_quality` obs columns
- `notebook_umap_by_subset.png`, `notebook_marker_heatmap.png`, `notebook_marker_dotplot.png`

To rerun on a different dataset, edit only the **Parameters** cell (Section 2) — everything else adapts automatically.